# ⚡ Delta-Compressed Embedding Engine (DCEE)

**GPU-native** vector index that exploits embedding correlation via delta encoding — like video compression applied to embeddings.

```
[K0] [Δ1] [Δ2] [Δ3] [K1] [Δ4] [Δ5] ...
E1 = K0 + Δ1
E2 = E1 + Δ2
```

**Runtime → Change runtime type → T4 GPU** before running!


In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────
!pip install -q cupy-cuda12x scikit-learn tqdm numpy
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: GPU check ────────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'No GPU detected'

In [ ]:
# ── Cell 3: Full DCEE Implementation ────────────────────────────────
"""
╔══════════════════════════════════════════════════════════════════════╗
║       Delta-Compressed Embedding Engine (DCEE) — GPU Native         ║
╚══════════════════════════════════════════════════════════════════════╝
ARCHITECTURE:
  [Cluster] → [Order] → [Delta Encode] → [Quantize] → [Binary Store]
  Query: Keyframe Search → Partial Reconstruct → Refine Top-K
"""

import os, time, struct, warnings
import numpy as np
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# GPU imports with graceful CPU fallback
try:
    import cupy as cp
    GPU_AVAILABLE = cp.cuda.is_available()
except ImportError:
    GPU_AVAILABLE = False
    print('⚠ CuPy not found — falling back to NumPy (CPU)')

if GPU_AVAILABLE:
    xp = cp
    print(f'✅ GPU: {cp.cuda.runtime.getDeviceProperties(0)["name"].decode()}')
else:
    xp = np
    print('ℹ  Running on CPU (NumPy)')

from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm
import pickle


# ═══════════════════════════════════════════════════════════════════════
# §1  DATA STRUCTURES
# ═══════════════════════════════════════════════════════════════════════

@dataclass
class DCEEConfig:
    """All tunable hyper-parameters."""
    dim: int = 128
    n_clusters: int = 64
    keyframe_every: int = 8
    quantization: str = 'int8'      # 'float16' | 'int8' | 'sparse'
    sparse_threshold: float = 0.02
    top_k_refine: int = 10
    batch_size: int = 1024

@dataclass
class ClusterBlock:
    cluster_id: int
    keyframe: np.ndarray
    deltas: list
    original_indices: List[int]
    keyframe_positions: List[int]

@dataclass
class DCEEIndex:
    config: DCEEConfig
    clusters: List[ClusterBlock]
    keyframe_matrix: np.ndarray
    cluster_norms: np.ndarray


# ═══════════════════════════════════════════════════════════════════════
# §2  CLUSTERING & ORDERING
# ═══════════════════════════════════════════════════════════════════════

class EmbeddingPreprocessor:
    def __init__(self, cfg):
        self.cfg = cfg

    def cluster(self, embeddings):
        print(f'\n📦 Clustering {len(embeddings):,} embeddings → {self.cfg.n_clusters} clusters …')
        normed = normalize(embeddings.astype(np.float32))
        km = MiniBatchKMeans(
            n_clusters=self.cfg.n_clusters,
            batch_size=max(4096, self.cfg.batch_size),
            n_init=3, random_state=42
        )
        labels = km.fit_predict(normed)
        groups = [[] for _ in range(self.cfg.n_clusters)]
        for i, lbl in enumerate(labels):
            groups[lbl].append(i)
        sizes = [len(g) for g in groups if g]
        print(f'   sizes — min:{min(sizes)}  max:{max(sizes)}  avg:{np.mean(sizes):.1f}')
        return groups

    def greedy_order(self, vecs):
        n = len(vecs)
        if n <= 2:
            return list(range(n))
        visited = np.zeros(n, dtype=bool)
        order = [0]
        visited[0] = True
        cur = vecs[0]
        for _ in range(n - 1):
            dists = np.sum((vecs - cur) ** 2, axis=1)
            dists[visited] = np.inf
            nxt = int(np.argmin(dists))
            order.append(nxt)
            visited[nxt] = True
            cur = vecs[nxt]
        return order


# ═══════════════════════════════════════════════════════════════════════
# §3  DELTA ENCODING + QUANTIZATION
# ═══════════════════════════════════════════════════════════════════════

class DeltaEncoder:
    def __init__(self, cfg):
        self.cfg = cfg

    def _to_float16(self, d):
        return d.astype(np.float16)

    def _to_int8(self, d):
        scale = np.max(np.abs(d)) / 127.0 + 1e-9
        return np.clip(np.round(d / scale), -127, 127).astype(np.int8), scale

    def _to_sparse(self, d):
        mask = np.abs(d) > self.cfg.sparse_threshold
        return np.where(mask)[0].astype(np.int16), d[mask].astype(np.float16)

    def _encode_delta(self, d):
        q = self.cfg.quantization
        if q == 'float16': return self._to_float16(d)
        if q == 'int8':    return self._to_int8(d)
        if q == 'sparse':  return self._to_sparse(d)
        return d.astype(np.float32)

    def decode_delta(self, encoded):
        q = self.cfg.quantization
        if q == 'float16': return encoded.astype(np.float32)
        if q == 'int8':    arr, scale = encoded; return arr.astype(np.float32) * scale
        if q == 'sparse':
            idx, vals = encoded
            d = np.zeros(self.cfg.dim, np.float32)
            d[idx] = vals.astype(np.float32)
            return d
        return encoded.astype(np.float32)

    def encode_cluster(self, vecs, global_ids):
        n, dim = vecs.shape
        kfe = self.cfg.keyframe_every
        kf_positions, all_deltas = [], []
        prev = None
        for i in range(n):
            is_kf = (i % kfe == 0)
            if is_kf:
                kf_positions.append(i)
                prev = vecs[i].copy()
                if i > 0:
                    all_deltas.append(self._encode_delta(np.zeros(dim, np.float32)))
            else:
                all_deltas.append(self._encode_delta(vecs[i] - prev))
                prev = vecs[i].copy()
        return ClusterBlock(
            cluster_id=global_ids[0],
            keyframe=vecs[0].astype(np.float32),
            deltas=all_deltas,
            original_indices=global_ids,
            keyframe_positions=kf_positions,
        )

    def decode_vector(self, block, local_idx):
        cur = block.keyframe.copy()
        for i in range(1, local_idx + 1):
            delta = self.decode_delta(block.deltas[i - 1])
            if i in block.keyframe_positions:
                cur = block.keyframe.copy()
            else:
                cur = cur + delta
        return cur


# ═══════════════════════════════════════════════════════════════════════
# §4  GPU QUERY ENGINE
# ═══════════════════════════════════════════════════════════════════════

class GPUQueryEngine:
    def __init__(self, index, encoder):
        self.index   = index
        self.encoder = encoder
        self.cfg     = index.config
        self.kf_gpu  = xp.array(index.keyframe_matrix, dtype=xp.float32)
        print(f'⚡ GPU keyframe matrix: {self.kf_gpu.shape}  ({self.kf_gpu.nbytes/1024:.1f} KB)')

    def search(self, query, top_k=5):
        q_gpu  = xp.array(query, dtype=xp.float32)
        q_norm = q_gpu / (xp.linalg.norm(q_gpu) + 1e-9)

        # Phase 1: keyframe ANN
        kf_norm = self.kf_gpu / (xp.linalg.norm(self.kf_gpu, axis=1, keepdims=True) + 1e-9)
        kf_scores = xp.asnumpy(kf_norm @ q_norm) if GPU_AVAILABLE else kf_norm @ q_norm
        n_probe = min(max(top_k, 8), len(self.index.clusters))
        top_cids = np.argsort(kf_scores)[::-1][:n_probe]

        # Phase 2: partial reconstruction
        candidates = []
        for cid in top_cids:
            block = self.index.clusters[cid]
            candidates.extend(self._partial_score(block, q_norm))

        # Phase 3: full-precision refine
        candidates.sort(key=lambda x: -x[0])
        refined = []
        for approx, gidx in candidates[:self.cfg.top_k_refine]:
            cid, lidx = self._global_to_local(gidx)
            block = self.index.clusters[cid]
            vec = self.encoder.decode_vector(block, lidx)
            qn  = xp.asnumpy(q_norm) if GPU_AVAILABLE else q_norm
            score = float(np.dot(vec, qn) / (np.linalg.norm(vec) + 1e-9))
            refined.append((gidx, score))

        refined.sort(key=lambda x: -x[1])
        return refined[:top_k]

    def _partial_score(self, block, q_norm):
        results = []
        cur = xp.array(block.keyframe, dtype=xp.float32)
        for i in range(len(block.original_indices)):
            if i > 0:
                d = xp.array(self.encoder.decode_delta(block.deltas[i-1]), dtype=xp.float32)
                cur = (xp.array(block.keyframe, xp.float32) if i in block.keyframe_positions
                       else cur + d)
            # Early exit on first 32 dims
            if i > 0 and float(xp.dot(cur[:32], q_norm[:32])) < -0.5:
                continue
            score = float(xp.dot(cur / (xp.linalg.norm(cur) + 1e-9), q_norm))
            results.append((score, block.original_indices[i]))
        return results

    def _global_to_local(self, gidx):
        for cid, block in enumerate(self.index.clusters):
            if gidx in block.original_indices:
                return cid, block.original_indices.index(gidx)
        raise ValueError(f'Global index {gidx} not found')


# ═══════════════════════════════════════════════════════════════════════
# §5  BINARY STORAGE
# ═══════════════════════════════════════════════════════════════════════

MAGIC, VERSION = b'DCEE', 1

def save_index(index, path):
    with open(path, 'wb') as f:
        f.write(MAGIC)
        f.write(struct.pack('B', VERSION))
        f.write(struct.pack('III', index.config.dim, len(index.clusters),
                            sum(len(c.original_indices) for c in index.clusters)))
        f.write(index.keyframe_matrix.astype(np.float32).tobytes())
        for block in index.clusters:
            n, nkf = len(block.original_indices), len(block.keyframe_positions)
            f.write(struct.pack('III', block.cluster_id, n, nkf))
            f.write(block.keyframe.astype(np.float32).tobytes())
            f.write(np.array(block.keyframe_positions, np.int32).tobytes())
            f.write(np.array(block.original_indices, np.int32).tobytes())
            db = pickle.dumps(block.deltas)
            f.write(struct.pack('I', len(db))); f.write(db)
    print(f'💾 Saved → {path}  ({os.path.getsize(path)/1e6:.2f} MB)')

def load_index(path, cfg):
    with open(path, 'rb') as f:
        assert f.read(4) == MAGIC
        f.read(1)  # version
        dim, nc, nv = struct.unpack('III', f.read(12))
        kfm = np.frombuffer(f.read(nc*dim*4), np.float32).reshape(nc, dim).copy()
        clusters = []
        for _ in range(nc):
            cid, n, nkf = struct.unpack('III', f.read(12))
            kf = np.frombuffer(f.read(dim*4), np.float32).copy()
            kfp = list(np.frombuffer(f.read(nkf*4), np.int32))
            oid = list(np.frombuffer(f.read(n*4), np.int32))
            dlen = struct.unpack('I', f.read(4))[0]
            deltas = pickle.loads(f.read(dlen))
            clusters.append(ClusterBlock(cid, kf, deltas, oid, kfp))
    index = DCEEIndex(cfg, clusters, kfm, np.linalg.norm(kfm, axis=1))
    print(f'📂 Loaded → {path}  ({nv:,} vectors, {nc} clusters)')
    return index


# ═══════════════════════════════════════════════════════════════════════
# §6  HIGH-LEVEL ENGINE
# ═══════════════════════════════════════════════════════════════════════

class DCEEEngine:
    def __init__(self, cfg):
        self.cfg = cfg
        self.pre = EmbeddingPreprocessor(cfg)
        self.enc = DeltaEncoder(cfg)
        self.index = None
        self.qe    = None

    def build(self, embeddings):
        t0 = time.time()
        emb = embeddings.astype(np.float32)
        groups = self.pre.cluster(emb)
        print(f'\n🔧 Delta-encoding {self.cfg.n_clusters} clusters …')
        clusters, kf_list = [], []
        for cid, gids in enumerate(tqdm(groups, desc='Encoding')):
            if not gids: continue
            vecs = emb[gids]
            order = self.pre.greedy_order(vecs)
            vecs_o = vecs[order]
            gids_o = [gids[o] for o in order]
            block = self.enc.encode_cluster(vecs_o, gids_o)
            block.cluster_id = cid
            clusters.append(block)
            kf_list.append(block.keyframe)
        kfm = np.stack(kf_list)
        self.index = DCEEIndex(self.cfg, clusters, kfm, np.linalg.norm(kfm, axis=1))
        self.qe = GPUQueryEngine(self.index, self.enc)
        N = len(embeddings)
        bpv = {'float16':2,'int8':1,'sparse':0.5,'float32':4}.get(self.cfg.quantization,4)
        raw = N * self.cfg.dim * 4 / 1e6
        comp = N * self.cfg.dim * bpv / 1e6
        print(f'\n{'═'*44}')
        print(f'  Vectors     : {N:>10,}')
        print(f'  Raw         : {raw:>9.2f} MB')
        print(f'  Compressed  : {comp:>9.2f} MB  ({raw/comp:.1f}×)')
        print(f'  Build time  : {time.time()-t0:>9.2f} s')
        print(f'{'═'*44}\n')

    def search(self, query, top_k=5):
        assert self.qe, 'Call build() first'
        return self.qe.search(query, top_k)

    def save(self, path): save_index(self.index, path)

    def load(self, path):
        self.index = load_index(path, self.cfg)
        self.qe = GPUQueryEngine(self.index, self.enc)

print('✅ DCEE classes loaded successfully!')

In [ ]:
# ── Cell 4: Generate correlated synthetic embeddings ─────────────────
DIM        = 128
N          = 50_000
N_CLUSTERS = 64

print(f'🎲 Generating {N:,} correlated embeddings (dim={DIM}) …')
rng    = np.random.default_rng(42)
topics = rng.standard_normal((N_CLUSTERS, DIM)).astype(np.float32)
topics = topics / np.linalg.norm(topics, axis=1, keepdims=True)
labels = rng.integers(0, N_CLUSTERS, size=N)
noise  = rng.standard_normal((N, DIM)).astype(np.float32) * 0.15
embeddings = topics[labels] + noise
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
print(f'shape={embeddings.shape}  dtype={embeddings.dtype}  ||e[0]||={np.linalg.norm(embeddings[0]):.3f}')

In [ ]:
# ── Cell 5: Build DCEE index ─────────────────────────────────────────
cfg = DCEEConfig(
    dim           = DIM,
    n_clusters    = N_CLUSTERS,
    keyframe_every= 8,
    quantization  = 'int8',   # try: 'float16', 'int8', 'sparse', 'float32'
    top_k_refine  = 20,
)
engine = DCEEEngine(cfg)
engine.build(embeddings)

In [ ]:
# ── Cell 6: Search benchmark ──────────────────────────────────────────
TOP_K = 5
N_QUERIES = 100
query_ids = rng.integers(0, N, size=N_QUERIES)

latencies, hits = [], 0
for qid in tqdm(query_ids, desc='Querying'):
    q = embeddings[qid]
    t0 = time.perf_counter()
    results = engine.search(q, top_k=TOP_K)
    latencies.append((time.perf_counter() - t0) * 1000)
    if int(qid) in [r for r, _ in results]:
        hits += 1

print(f'\n{'─'*38}')
print(f'  Recall@{TOP_K}    : {hits/N_QUERIES*100:.1f}%')
print(f'  Latency p50 : {np.percentile(latencies,50):.2f} ms')
print(f'  Latency p95 : {np.percentile(latencies,95):.2f} ms')
print(f'  Latency p99 : {np.percentile(latencies,99):.2f} ms')
print(f'{'─'*38}')

In [ ]:
# ── Cell 7: Save and reload index ────────────────────────────────────
SAVE_PATH = '/tmp/dcee_demo.dcee'
engine.save(SAVE_PATH)

engine2 = DCEEEngine(cfg)
engine2.load(SAVE_PATH)
# Quick sanity check
q = embeddings[0]
r = engine2.search(q, top_k=3)
print(f'Sanity check — top-3 results: {r}')

In [ ]:
# ── Cell 8: Quantization mode comparison table ───────────────────────
print('📊 Quantization mode comparison\n')
print(f'{"Mode":<10} {"Build(s)":>10} {"Recall%":>9} {"P50ms":>8} {"Est.MB":>9}')
print('─' * 50)

for qmode in ['float32', 'float16', 'int8', 'sparse']:
    cfg_q = DCEEConfig(dim=DIM, n_clusters=N_CLUSTERS,
                       keyframe_every=8, quantization=qmode, top_k_refine=20)
    eng_q = DCEEEngine(cfg_q)
    t0 = time.time()
    eng_q.build(embeddings)
    build_t = time.time() - t0

    lats, rec = [], 0
    for qid in query_ids[:50]:
        q = embeddings[qid]
        t1 = time.perf_counter()
        res = eng_q.search(q, top_k=TOP_K)
        lats.append((time.perf_counter() - t1) * 1000)
        if int(qid) in [r for r, _ in res]: rec += 1

    bpv = {'float16':2,'int8':1,'sparse':0.5,'float32':4}[qmode]
    est_mb = N * DIM * bpv / 1e6
    print(f'{qmode:<10} {build_t:>10.2f} {rec/50*100:>8.1f}% {np.median(lats):>7.2f} {est_mb:>9.2f}')

In [ ]:
# ── Cell 9: Keyframe spacing effect ──────────────────────────────────
print('📊 Keyframe spacing vs compression/recall\n')
print(f'{"KF every":>9} {"Recall%":>9} {"P50ms":>8}')
print('─' * 30)

for kfe in [4, 8, 16, 32, 64]:
    cfg_k = DCEEConfig(dim=DIM, n_clusters=N_CLUSTERS,
                       keyframe_every=kfe, quantization='int8', top_k_refine=20)
    eng_k = DCEEEngine(cfg_k)
    eng_k.build(embeddings)
    lats, rec = [], 0
    for qid in query_ids[:50]:
        q = embeddings[qid]
        t1 = time.perf_counter()
        res = eng_k.search(q, top_k=TOP_K)
        lats.append((time.perf_counter() - t1) * 1000)
        if int(qid) in [r for r, _ in res]: rec += 1
    print(f'{kfe:>9} {rec/50*100:>8.1f}% {np.median(lats):>7.2f}')

## 🔌 Plug in your own embeddings

```python
# your_embeddings: np.ndarray shape (N, D) float32
cfg = DCEEConfig(
    dim=your_embeddings.shape[1],
    n_clusters=128,          # tune: ~sqrt(N)
    keyframe_every=8,        # tune: 4–32
    quantization='int8',     # 'float16' | 'int8' | 'sparse'
    top_k_refine=20,
)
engine = DCEEEngine(cfg)
engine.build(your_embeddings)
engine.save('my_index.dcee')

results = engine.search(query_vec, top_k=10)
# [(global_index, cosine_similarity), ...]
```

**Best datasets:** document chunks from same file, chat logs, time-series sensor data, semantic clusters.

**Rule of thumb:** `n_clusters ≈ sqrt(N)`, `keyframe_every = 8` is a good default.
